## Base line model :
- we will the raw data and some simple features like the hour ,day of the week,month 
- and the log of the trip_duration as our target variable 

In [57]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd 
import numpy as np
import os 
root_path='D:/Python/ML/projects/nyc-taxi-trip-duration/data/'

In [58]:
train=pd.read_csv(os.path.join(root_path,'train.csv'))
val=pd.read_csv(os.path.join(root_path,'val.csv'))

In [59]:
train.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2793718,2,2016-06-08 07:36:19,1,-73.985611,40.735943,-73.980331,40.760468,N,1040
1,id3485529,2,2016-04-03 12:58:11,1,-73.978394,40.764351,-73.991623,40.749859,N,827
2,id1816614,2,2016-06-05 02:49:13,5,-73.989059,40.744389,-73.973381,40.748692,N,614
3,id1050851,2,2016-05-05 17:18:27,2,-73.990326,40.731136,-73.991264,40.748917,N,867
4,id0140657,1,2016-05-12 17:43:38,4,-73.789497,40.646675,-73.987137,40.759232,N,4967


In [ ]:
def prepare_data(train):
    train.drop(columns=['id'], inplace=True)

    train['pickup_datetime'] = pd.to_datetime(train['pickup_datetime'])
    train['dayofweek'] = train.pickup_datetime.dt.dayofweek
    train['month'] = train.pickup_datetime.dt.month
    train['hour'] = train.pickup_datetime.dt.hour
    train['dayofyear']  = train.pickup_datetime.dt.dayofyear
    train['trip_duration']=np.log1p(train['trip_duration'])
    return train
    


In [61]:
train=prepare_data(train)
val=prepare_data(val)

In [62]:
numeric_features = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']
categorical_features = ['dayofweek', 'month', 'hour', 'dayofyear', 'passenger_count']
train_features = categorical_features + numeric_features

In [63]:
column_transformer = ColumnTransformer([
('ohe', OneHotEncoder(handle_unknown="ignore"), categorical_features),
('scaling', StandardScaler(), numeric_features)
]
, remainder = 'passthrough'
)

In [64]:
pipe = Pipeline(steps=[
('ohe', column_transformer),
('regression', Ridge())
    ])

In [65]:
model= pipe.fit(train[train_features], train['trip_duration'])

In [66]:
def model_eval(model,x,t,mesg='none'):
    pred=model.predict(x)
    mse=mean_squared_error(t,pred)
    r2=r2_score(t,pred)
    print(f'{mesg} evaluation')
    print(f"MSE: {mse}")
    print(f"R2 score: {r2}")
    print('================================================')

In [67]:
model_eval(pipe,train[train_features],train.trip_duration,'train')

train evaluation
MSE: 0.5910464642065324
R2 score: 0.06423270499770728


In [68]:
model_eval(pipe,val[train_features],val.trip_duration,'Validation')

Validation evaluation
MSE: 0.5975833295108223
R2 score: 0.06632946180334198


## Why didn't this model work?

- As expected, the baseline model performed poorly with an R2 score of 0.066. Because it only has access to raw GPS coordinates, the model does not understand the layout of NYC To fix this, I will move to the main pipeline and engineer temporal features and geospatial K-Means clusters.